Selección y Validación de Modelos
INF-8267-C1    

**Asignación Semana 2- ¿Podemos creer nuestras métricas?**

#### **1- Revisar la partición definida en Semana 1 y justificar por qué representa producción.**

#### **2- Verificar que no exista fuga de información en:**  
*   variables;  
*   partición;



> Por: Rosanny Batista Rodríguez






In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Cargamos el dataset y los archivos de la partición de la asignación Semana 1
from google.colab import files
import io

print("Sube el dataset: 'constructora_horizonte_dias_restantes.csv' y los archivos: 'train.csv', 'validacion.csv', y 'test.csv'")
archivos = files.upload()

Sube el dataset: 'constructora_horizonte_dias_restantes.csv' y los archivos: 'train.csv', 'validacion.csv', y 'test.csv'


Saving constructora_horizonte_dias_restantes.csv to constructora_horizonte_dias_restantes.csv
Saving test.csv to test.csv
Saving train.csv to train.csv
Saving validacion.csv to validacion.csv


In [ ]:
#Carga de datos

RUTA_DATASET = "constructora_horizonte_dias_restantes.csv"

# Particiones ya generadas en la Semana 1.
# Estas ya tienen las columnas de fuga eliminadas
RUTA_TRAIN = "train.csv"
RUTA_VAL = "validacion.csv"
RUTA_TEST = "test.csv"

COL_FECHA = "week_date" #tiene la fecha de cada registro
COL_ENTIDAD = "workfront_id" #identifica a cada frente de trabajo
COL_FECHA_FIN = "final_completion_date" #fecha real en que el frente terminó

# Definidas en la auditoría de Semana 1 como posteriores al momento de predicción
VARIABLES_FUGA = [
    "final_completion_date", "total_final_cost",
    "final_delay_reason", "final_progress_report",
]

dataset = pd.read_csv(io.BytesIO(archivos[RUTA_DATASET]), parse_dates=[COL_FECHA, COL_FECHA_FIN])
train = pd.read_csv(io.BytesIO(archivos[RUTA_TRAIN]), parse_dates=[COL_FECHA])
validacion = pd.read_csv(io.BytesIO(archivos[RUTA_VAL]), parse_dates=[COL_FECHA])
test = pd.read_csv(io.BytesIO(archivos[RUTA_TEST]), parse_dates=[COL_FECHA])

print(f"\nDataset: {len(dataset):,} filas, {dataset[COL_ENTIDAD].nunique()} frentes")
for nombre, df in [("train", train), ("validacion", validacion), ("test", test)]:
    print(f"{nombre}: {len(df):,} filas, {df[COL_ENTIDAD].nunique()} frentes")


Dataset: 5,728 filas, 150 frentes
train: 2,805 filas, 150 frentes
validacion: 1,092 filas, 128 frentes
test: 1,273 filas, 119 frentes


## 1- ¿La partición representa producción?

In [ ]:
# Revisar la partición definida en Semana 1 y justificar por qué representa producción.

print("\nVerificacion 1: orden cronologico entre conjuntos (el resultado de esa partición)")
print(f"Train:      {train[COL_FECHA].min().date()} a {train[COL_FECHA].max().date()}")
print(f"Validacion: {validacion[COL_FECHA].min().date()} a {validacion[COL_FECHA].max().date()}")
print(f"Test:       {test[COL_FECHA].min().date()} a {test[COL_FECHA].max().date()}")

orden_correcto = (
    train[COL_FECHA].max() < validacion[COL_FECHA].min() and
    validacion[COL_FECHA].max() < test[COL_FECHA].min()
)
print(f"Orden cronologico sin solapamiento entre conjuntos: {orden_correcto}")

# Verificacion 2:
print("\n Verificacion 2: entidades que aparecen en mas de un conjunto")
entidades_train = set(train[COL_ENTIDAD].unique())
entidades_test = set(test[COL_ENTIDAD].unique())
interseccion = entidades_train & entidades_test
print(f"Frentes que aparecen en train Y en test: {len(interseccion)} de {len(entidades_test)} "
      f"({len(interseccion)/len(entidades_test):.1%})")
print("La interpretacion de este numero depende de si la partición pretende evaluar, frentes ya conocidos (caso de este dataset) o generalización a frentes nuevos.")
print("La verificacion 3 evalua si, dentro de un mismo frente, hay mezcla de semanas entre train y test sin respetar el orden temporal.")

# Verificacion 3:
print("\nVerificacion 3: orden temporal dentro de cada frente compartido")
violaciones = 0
for frente in interseccion:
    max_fecha_train = train.loc[train[COL_ENTIDAD] == frente, COL_FECHA].max()
    min_fecha_test = test.loc[test[COL_ENTIDAD] == frente, COL_FECHA].min()
    if max_fecha_train >= min_fecha_test:
        violaciones += 1
print(f"Frentes donde una semana de train es igual o posterior a una semana de test: {violaciones} "
      f"de {len(interseccion)}")


Verificacion 1: orden cronologico entre conjuntos (el resultado de esa partición)
Train:      2025-01-19 a 2025-06-30
Validacion: 2025-07-01 a 2025-08-31
Test:       2025-09-01 a 2025-12-11
Orden cronologico sin solapamiento entre conjuntos: True

 Verificacion 2: entidades que aparecen en mas de un conjunto
Frentes que aparecen en train Y en test: 119 de 119 (100.0%)
La interpretacion de este numero depende de si la partición pretende evaluar, frentes ya conocidos (caso de este dataset) o generalización a frentes nuevos.
La verificacion 3 evalua si, dentro de un mismo frente, hay mezcla de semanas entre train y test sin respetar el orden temporal.

Verificacion 3: orden temporal dentro de cada frente compartido
Frentes donde una semana de train es igual o posterior a una semana de test: 0 de 119


# 2 - Verificar que no exista fuga de información en:

## Variables
Se revisa si las variables clasificadas como 'posteriores al momento de prediccion' en la auditoria (Semana 1) estan presentes en los archivos usados para entrenar y evaluar el modelo.

In [ ]:
print("Verificación de fuga en variables:")

for nombre, df in [("train", train), ("validacion", validacion), ("test", test)]:
    columnas_de_fuga_presentes = [c for c in VARIABLES_FUGA if c in df.columns]
    print(f"{nombre}: variables de fuga presentes en el archivo: {columnas_de_fuga_presentes or 'ninguna'}")

Verificación de fuga en variables:
train: variables de fuga presentes en el archivo: ninguna
validacion: variables de fuga presentes en el archivo: ninguna
test: variables de fuga presentes en el archivo: ninguna




> Adicionalmente, se revisa el uso de engineer_estimate_days: esta variable si esta presente en los archivos (a diferencia de las 4 anteriores), por lo que su clasificacion como fuga o no depende de como se use en el modelado, no de su sola presencia. Ver notebook de modelado para confirmar que se usa unicamente como baseline y no como feature de entrada.



## Fuga en la partición

Se reconstruye final_completion_date desde el dataset crudo (por workfront_id) y se compara contra week_date de cada fila ya particionada, para contar cuantas filas quedaron registradas despues de la fecha de finalizacion de su propio frente.


In [ ]:
print("Verificación de fuga en la partición:")

fin_real = dataset.groupby(COL_ENTIDAD)[COL_FECHA_FIN].first()

resumen_fuga = []
for nombre, df in [("train", train), ("validacion", validacion), ("test", test)]:
    parte = df.merge(fin_real, on=COL_ENTIDAD, how="left")
    filas_fuga = (parte[COL_FECHA] > parte[COL_FECHA_FIN]).sum()
    pct = filas_fuga / len(parte) * 100
    resumen_fuga.append({"conjunto": nombre, "filas_totales": len(parte),
                          "filas_con_fuga": int(filas_fuga), "pct_con_fuga": round(pct, 2)})
    print(f"{nombre}: {len(parte):,} filas totales, {filas_fuga} filas con fuga ({pct:.2f}%)")

resumen_fuga_df = pd.DataFrame(resumen_fuga)


Verificación de fuga en la partición:
train: 2,805 filas totales, 90 filas con fuga (3.21%)
validacion: 1,092 filas totales, 58 filas con fuga (5.31%)
test: 1,273 filas totales, 113 filas con fuga (8.88%)




> Para entender por que estas filas no fueron descartadas, se revisa el script de particion: incluye un filtro pensado para eliminar filas posteriores a la finalizacion del frente, condicionado a que la columna final_completion_date este presente en el dataframe. En el bloque principal de ese script, sin embargo, las columnas de fuga se eliminan ANTES de invocar la funcion que aplica ese filtro, por lo que la condicion nunca se cumple y el filtro no llega a ejecutarse sobre ningun conjunto.



In [ ]:
#Se guarda la evidencia
resumen_fuga_df.to_csv("evidencia_verificacion_fuga_particion.csv", index=False)
print("\nGuardado: evidencia_verificacion_fuga_particion.csv")


Guardado: evidencia_verificacion_fuga_particion.csv
